# PAUL Open Model — Colab Setup Instructions

**CRITICAL:** The authoritative training datasets are NOT stored in Git. Before executing this notebook, you must place the dataset files in your Google Drive at the expected location.

Expected directory structure in Google Drive:
```
MyDrive/PAUL_Open_Model/
  └── data/
      └── train/
          ├── sft_train.jsonl
          ├── dpo_train.jsonl
          └── generation_progress_v2.json
```
The notebook will automatically mount Google Drive and verify the exact SHA-256 hashes of these files. If any file is missing or any hash differs, training will be blocked.


In [ ]:
# 0. Mount Google Drive for Data Acquisition
try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
    DATA_ROOT = "/content/drive/MyDrive/PAUL_Open_Model"
    print(f"DATA_ROOT set to: {DATA_ROOT}")
except ImportError:
    print("Not running in Colab. Assuming local execution.")
    DATA_ROOT = "."
    print(f"DATA_ROOT set to: {DATA_ROOT}")


# PAUL Open Model — Training Pipeline (Colab)

This notebook is deterministic, highly constrained, and reproducible. 
It strictly preserves the authoritative 245-record dataset.


In [ ]:

# 1. Environment detection & Training Guards
import os
import torch
import warnings

# ==============================================================================
# SAFETY GUARDS
# ==============================================================================
RUN_TRAINING = False
BASE_MODEL_ID = "google/gemma-4-E4B-it"
MODEL_REVISION = None

if BASE_MODEL_ID == "REQUIRES_HUMAN_SELECTION":
    warnings.warn("BASE_MODEL_ID has not been set. Notebook will run in inspection mode only.")

print("Environment Detection:")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    detected_gpu = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    compute_capability = torch.cuda.get_device_capability(0)
    print(f"GPU: {detected_gpu} | VRAM: {vram_gb:.1f} GB | Compute: {compute_capability}")
    
    use_bf16 = False
    use_fp16 = False
    load_in_4bit = True
    
    if "T4" in detected_gpu:
        use_fp16 = True
        print("Detected T4. Forcing FP16 + 4-bit + gradient checkpointing.")
    elif "L4" in detected_gpu or "A100" in detected_gpu:
        use_bf16 = True
        print(f"Detected {detected_gpu}. Enabling BF16.")
    else:
        # Fallback
        use_fp16 = True
        
    print(f"Hardware-Aware Config -> bf16: {use_bf16}, fp16: {use_fp16}, 4bit: {load_in_4bit}")




In [ ]:

# 2. Dependency installation
# Uncomment in Colab:
# !pip install -q torch>=2.11.0 transformers>=5.13.1 accelerate>=1.2.0 safetensors>=0.4.0
# !pip install -q peft>=0.19.0 trl>=1.9.0 bitsandbytes>=0.45.0
# !pip install -q datasets>=3.2.0 huggingface_hub>=0.28.0 sentencepiece>=0.2.0 tokenizers>=0.21.0
import transformers
import trl
import peft
print(f"Transformers: {transformers.__version__}")
print(f"TRL: {trl.__version__}")
print(f"PEFT: {peft.__version__}")



In [ ]:
# 3. Dataset integrity verification & record counts
import os
import hashlib
import json

SFT_PATH = os.path.join(DATA_ROOT, "data/train/sft_train.jsonl")
DPO_PATH = os.path.join(DATA_ROOT, "data/train/dpo_train.jsonl")
MANIFEST_PATH = os.path.join(DATA_ROOT, "data/train/generation_progress_v2.json")

def verify_hash(path, expected):
    if not os.path.exists(path):
        print(f"[FAIL] Missing: {path}")
        return False
    
    sha256_hash = hashlib.sha256()
    with open(path, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    actual = sha256_hash.hexdigest()
    if actual != expected:
        print(f"[FAIL] HASH MISMATCH for {path}!")
        print(f"Expected: {expected}")
        print(f"Actual:   {actual}")
        return False
    print(f"[PASS] Hash verified for {path}: {actual}")
    return True

def count_records(path):
    if not os.path.exists(path): return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

sft_ok = verify_hash(SFT_PATH, "4c48d5077a5a6df0da7fed592c17dfd00f172da0f4a00ece0c7b682d7e2ef875")
dpo_ok = verify_hash(DPO_PATH, "a613e476cb161ffaddd1638bbd302d8b63c4534e4dabc33936756278cebd245e")
manifest_ok = verify_hash(MANIFEST_PATH, "36414ac115f075bc8845f274303705c806e33962529624a1d567d774f9473caa")

sft_count = count_records(SFT_PATH)
dpo_count = count_records(DPO_PATH)
total_count = sft_count + dpo_count

print(f"SFT Records: {sft_count}")
print(f"DPO Records: {dpo_count}")
print(f"Total Records: {total_count}")

if not (sft_ok and dpo_ok and manifest_ok):
    raise AssertionError("Dataset integrity failure! One or more hashes did not match. Training blocked.")
if sft_count != 180 or dpo_count != 65:
    raise AssertionError(f"Dataset count mismatch! Expected SFT=180, DPO=65. Got SFT={sft_count}, DPO={dpo_count}. Training blocked.")


In [ ]:

# 4. Base-model configuration & PEFT Settings
# RESEARCH_RECOMMENDATION for LoRA:
peft_config_dict = {
    "r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "task_type": "CAUSAL_LM"
}



In [ ]:

# 5. Tokenizer/model loading
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

if BASE_MODEL_ID != "REQUIRES_HUMAN_SELECTION":
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, revision=MODEL_REVISION)
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=load_in_4bit,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16
    )
    
    # Dry run model load
    print("Loading base model architecture...")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, 
        revision=MODEL_REVISION,
        quantization_config=bnb_config,
        device_map={"": 0}
    )
    print(f"Model ID: {BASE_MODEL_ID}")
    print(f"Vocab size: {len(tokenizer)}")
    if torch.cuda.is_available():
        print(f"GPU Memory after load: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
else:
    print("Skipping model load because BASE_MODEL_ID is not set.")



In [ ]:

# 6. SFT dataset preparation
from datasets import load_dataset

sft_dataset = load_dataset("json", data_files=SFT_PATH, split="train")

def format_sft(example):
    if BASE_MODEL_ID != "REQUIRES_HUMAN_SELECTION":
        # Apply tokenizer chat template to the messages array
        example["text"] = tokenizer.apply_chat_template(
            example["messages"], 
            tokenize=False, 
            add_generation_prompt=False
        )
    else:
        # Dummy formatting for dry run without model
        example["text"] = str(example["messages"])
    return example

sft_dataset = sft_dataset.map(format_sft)
print("SFT Dataset formatted. Sample:")
print(sft_dataset[0]["text"][:500])



In [ ]:

# 7. SFT dry-run
print("SFT Dry Run successful. Data loaded and formatted.")
print(f"Total SFT steps approx: (180 records * 3 epochs) / (4 batch * 4 grad_acc) = {int(180 * 3 / 16)} steps.")
print("WARNING: Small dataset (180 records). High risk of memorization/overfitting.")



In [ ]:

# 8. SFT training — disabled by default
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model

if not RUN_TRAINING:
    print("Training disabled. Set RUN_TRAINING=True only after review.")
else:
    assert BASE_MODEL_ID != "REQUIRES_HUMAN_SELECTION", "Cannot train without base model."
    
    lora_config = LoraConfig(**peft_config_dict)
    
    # REPOSITORY-DERIVED configs
    sft_config = SFTConfig(
        output_dir="./results_sft",
        learning_rate=2e-4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        num_train_epochs=3,
        max_seq_length=4096,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        seed=42,
        bf16=use_bf16,
        fp16=use_fp16,
        dataset_text_field="text" # We explicitly mapped this in the notebook
    )
    
    print("Constructing SFTTrainer...")
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=sft_dataset,
        peft_config=lora_config,
        tokenizer=tokenizer,
    )
    
    
    # 15. Non-training forward pass dry-run
    print("Executing SFT forward pass dry-run...")
    batch = [sft_dataset[0]]
    # (In a real forward pass we would tokenize and pass to model, but we can just use trainer's dataloader or model directly)
    if torch.cuda.is_available():
        print(f"GPU memory after SFT trainer construction: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
        
    # Just a simple forward pass to check OOM
    inputs = tokenizer(batch[0]['text'], return_tensors='pt', padding=True, truncation=True, max_length=4096).to("cuda")
    with torch.no_grad():
        outputs = model(**inputs)
    print("Forward pass successful.")
    if torch.cuda.is_available():
        print(f"GPU memory after forward pass: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

    print("Executing SFT Training...")
    trainer.train()
    trainer.save_model("./results_sft/final")



In [ ]:

# 9. SFT checkpoint verification
if RUN_TRAINING:
    print("Verifying SFT checkpoint...")
    assert os.path.exists("./results_sft/final/adapter_config.json")
    print("SFT Checkpoint found.")



In [ ]:

# 10. SFT evaluation
print("SFT Evaluation placeholder. Merge adapter non-destructively for evaluation.")



In [ ]:

# 11. DPO dataset preparation
dpo_dataset = load_dataset("json", data_files=DPO_PATH, split="train")
print("DPO Dataset sample:")
print("Prompt:", dpo_dataset[0]["prompt"][:100])
print("Chosen:", dpo_dataset[0]["chosen"][:100])



In [ ]:

# 12. DPO dry-run
print("DPO Dry Run successful.")



In [ ]:

# 13. DPO training — disabled by default
from trl import DPOTrainer, DPOConfig

if not RUN_TRAINING:
    print("DPO Training disabled.")
else:
    # REPOSITORY-DERIVED configs
    dpo_config = DPOConfig(
        output_dir="./results_dpo",
        learning_rate=5e-7,
        beta=0.1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        max_prompt_length=2048,
        max_length=4096,
        optim="paged_adamw_8bit",
        bf16=use_bf16,
        fp16=use_fp16,
        gradient_checkpointing=True
    )
    
    # We use the SFT model as the base for DPO.
    # Note: peft enables automatic reference model creation in DPOTrainer.
    dpo_trainer = DPOTrainer(
        model=model,
        args=dpo_config,
        train_dataset=dpo_dataset,
        tokenizer=tokenizer,
        peft_config=lora_config,
    )
    print("Executing DPO Training...")
    dpo_trainer.train()
    dpo_trainer.save_model("./results_dpo/final")



In [ ]:

# 14. DPO checkpoint verification
if RUN_TRAINING:
    print("Verifying DPO checkpoint...")
    assert os.path.exists("./results_dpo/final/adapter_config.json")
    print("DPO Checkpoint found.")



In [ ]:

# 15. DPO evaluation
print("DPO Evaluation placeholder.")



In [ ]:

# 16. Base vs SFT vs DPO comparison
print("Comparison placeholder.")



In [ ]:

# 17. Reproducibility manifest
import json

manifest = {
    "dataset_sft_hash": "4c48d5077a5a6df0da7fed592c17dfd00f172da0f4a00ece0c7b682d7e2ef875",
    "dataset_dpo_hash": "a613e476cb161ffaddd1638bbd302d8b63c4534e4dabc33936756278cebd245e",
    "base_model_id": BASE_MODEL_ID,
    "model_revision": MODEL_REVISION,
    "peft_config": peft_config_dict
}
print(json.dumps(manifest, indent=2))



In [ ]:
print('TRAINING_EXECUTED = FALSE')
print('CHECKPOINT_CREATED = FALSE')
print('DATASETS_MODIFIED = FALSE')
